# AIGoat on Google Colab

Runs [AIGoat](https://github.com/AISecurityConsortium/AIGoat) end-to-end inside a Colab runtime. Installs Ollama, clones the repo, installs Python + npm deps, seeds the SQLite DB, starts the FastAPI backend and React frontend in the background, and exposes the app through Colab's built-in port proxy.

## Before you run

- **Runtime**: the default CPU runtime works. A GPU runtime (T4) makes Mistral dramatically faster — switch via `Runtime → Change runtime type` before running.
- **RAM**: you need at least ~12 GB free. Mistral alone is ~4.5 GB; embeddings + backend + Node dev server take another few GB.
- **First run is slow.** Expect ~3–5 min on the `pip install` cell and another several minutes on `ollama pull mistral` (it's a 4.5 GB download).
- **CPU inference is slow.** If you're on CPU-only, chat responses may take 30–60 seconds each.
- **Single-port proxy.** Colab can only realistically expose one port, so the frontend is patched at runtime to use relative URLs and the React dev server proxies all API calls to the backend. You will only open the `3000` URL.

Run the cells top-to-bottom. The last cell prints the clickable link.

## 1. Install Ollama and Serve Ollama

In [17]:
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 3,917 B in 1s (2,919 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [22]:
import subprocess
import time

# Start ollama serve in the background
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Wait for the server to start

## 2. Clone AIGoat and set the working directory

`os.chdir` is used instead of `!cd` so subsequent Python cells inherit the directory.

In [23]:
import os, subprocess

REPO_DIR = '/content/AIGoat'
REPO_URL = 'https://github.com/AISecurityConsortium/AIGoat.git'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.makedirs('logs', exist_ok=True)

# Make the repo importable as a package (so 'from app.core.database import ...' works later).
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('cwd:', os.getcwd())

cwd: /content/AIGoat


## 3. Install Python dependencies

No virtualenv — Colab already has Python 3.11. This step is slow on first run because `sentence-transformers`, `chromadb`, and `nemoguardrails` pull in a lot of transitive packages.

In [24]:
!ollama list
!pip install -r requirements.txt

NAME    ID    SIZE    MODIFIED 


## 4. Patch the frontend to use relative API URLs

`frontend/src/config/api.js` hardcodes `http://localhost:8000` as the fallback when `REACT_APP_API_URL` isn't set. Your browser is talking to a Colab proxy URL, not to localhost, so we rewrite the fallback to an empty string. That makes axios use relative URLs, which are sent to the React dev server on port 3000, which then forwards `/api/…` to `http://localhost:8000` via the `"proxy"` field already set in `frontend/package.json`. Net effect: one exposed port, everything works.

In [25]:
from pathlib import Path

p = Path('/content/AIGoat/frontend/src/config/api.js')
src = p.read_text()

# Target string to be replaced
target_str = "process.env.REACT_APP_API_URL || 'http://localhost:8000'"
# Desired replacement string
replacement_str = "''"

# Check if the file is already in the desired state
# The original code's replacement results in `BASE_URL: ''`
if f"BASE_URL: {replacement_str}," in src:
    print('api.js is already patched.')
elif target_str in src:
    patched = src.replace(target_str, replacement_str)
    p.write_text(patched)
    print('Patched', p)
else:
    # If the target string is not found and the file isn't already patched,
    # it means the repo layout changed in an unexpected way.
    raise RuntimeError('api.js default URL not found — repo layout may have changed unexpectedly.')

api.js is already patched.


## 5. Install frontend npm packages

Using `%%bash` so the `cd` is scoped to this cell.

In [26]:
%%bash
cd /content/AIGoat/frontend
npm install --silent --no-audit --no-fund 2>&1 | tail -20

## 6. Start the Ollama daemon (background)

`subprocess.Popen` keeps the daemon alive for the whole kernel session. A plain `!ollama serve &` would be killed when the cell finishes.

In [27]:
import subprocess, time
import urllib.request, urllib.error

ollama_log = open('/content/AIGoat/logs/ollama.log', 'wb')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2) as r:
            if r.status == 200:
                print('Ollama is up on :11434')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama did not start — check logs/ollama.log')

Ollama is up on :11434


## 7. Pull the Mistral model

~4.5 GB download on first run. Output streams progress bars.

In [28]:
!ollama pull mistral
!ollama list


NAME              ID              SIZE      MODIFIED               
mistral:latest    6577803aa9a0    4.4 GB    Less than a second ago    


## 8. Initialize the database

In [29]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from app.core.database import init_db

asyncio.run(init_db())
print('DB initialized')


DB initialized


## 9. Seed demo data (idempotent)

Matches the counts the bash script checks for: 5 users, 20 products, 9 challenges.

In [30]:
import asyncio, subprocess
from sqlalchemy import select, func
from app.core.database import async_session
from app.models import User, Product, Challenge

async def _needs_seed():
    async with async_session() as db:
        u = (await db.execute(select(func.count(User.id)))).scalar() or 0
        p = (await db.execute(select(func.count(Product.id)))).scalar() or 0
        c = (await db.execute(select(func.count(Challenge.id)))).scalar() or 0
        return u < 5 or p < 20 or c < 9

if asyncio.run(_needs_seed()):
    subprocess.run(['python', '-m', 'scripts.seed'], check=True, cwd='/content/AIGoat')
    print('Demo data seeded')
else:
    print('Database already has required data')

Demo data seeded


## 10. Start the backend (FastAPI / uvicorn) in the background

In [31]:
import subprocess, time
import urllib.request, urllib.error

backend_log = open('/content/AIGoat/logs/backend.log', 'wb')
backend_proc = subprocess.Popen(
    [
        'python', '-m', 'uvicorn', 'app.main:app',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--log-level', 'info',
    ],
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    cwd='/content/AIGoat',
)

for _ in range(60):
    try:
        with urllib.request.urlopen('http://localhost:8000/docs', timeout=2) as r:
            if r.status < 500:
                print('Backend is up on :8000')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    if backend_proc.poll() is not None:
        raise RuntimeError('Backend exited early — run the logs cell at the bottom to see why')
    time.sleep(1)
else:
    print('Backend slow to start — check logs/backend.log if the next cells fail')

Backend is up on :8000


## 11. Start the frontend (React dev server) in the background

`CI=true` suppresses CRA's interactive "port in use" prompt and disables the auto-browser-open.

In [32]:
import os, subprocess, time
import urllib.request, urllib.error

env = os.environ.copy()
env['PORT'] = '3000'
env['HOST'] = '0.0.0.0'
env['BROWSER'] = 'none'
env['CI'] = 'true'
env['DANGEROUSLY_DISABLE_HOST_CHECK'] = 'true'
env['WDS_SOCKET_PORT'] = '0'

frontend_log = open('/content/AIGoat/logs/frontend.log', 'wb')
frontend_proc = subprocess.Popen(
    ['npm', 'start'],
    stdout=frontend_log,
    stderr=subprocess.STDOUT,
    cwd='/content/AIGoat/frontend',
    env=env,
)

# CRA cold compile is slow — wait up to ~3 minutes.
for _ in range(180):
    try:
        with urllib.request.urlopen('http://localhost:3000/', timeout=2) as r:
            if r.status == 200:
                print('Frontend is up on :3000')
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    if frontend_proc.poll() is not None:
        raise RuntimeError('Frontend exited early — run the logs cell at the bottom to see why')
    time.sleep(1)
else:
    print('Frontend slow to compile — check logs/frontend.log if the preview URL 404s')

Frontend is up on :3000


## 12. Open AIGoat in a browser tab

`serve_kernel_port_as_window` prints a clickable proxy URL that forwards to `localhost:3000`. All `/api/…` requests sent to that URL are routed to the backend by the React dev server's proxy config, so the same URL also serves `/docs` (FastAPI Swagger).

In [34]:
from google.colab import output

output.serve_kernel_port_as_window(3000)

print()
print('==========================================')
print('       AI Goat is running!                ')
print('==========================================')
print()
print('  Click the link above to open the app.')
print('  API docs: append /docs to that URL')
print()
print('  Demo login:  alice / password123')
print('  Admin login: admin / admin123')
print()

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


       AI Goat is running!                

  Click the link above to open the app.
  API docs: append /docs to that URL

  Demo login:  alice / password123
  Admin login: admin / admin123



## 13. (Optional) Tail logs or shut everything down

Uncomment whatever you need. Process handles (`ollama_proc`, `backend_proc`, `frontend_proc`) are held in kernel memory from the earlier cells.

In [ ]:
# !tail -n 80 /content/AIGoat/logs/backend.log
# !tail -n 80 /content/AIGoat/logs/frontend.log
# !tail -n 80 /content/AIGoat/logs/ollama.log

#frontend_proc.terminate()
# backend_proc.terminate()
# ollama_proc.terminate()